In [1]:
import pandas as pd
import numpy as np

In [10]:
from pathlib import Path

stand = pd.read_csv("data/pre-thin-data.csv")   
xy    = pd.read_csv("data/dillwyn_xy_clean.csv")             

ROW, TREE = "Row", "Tree"

def basic_report(df, name):
    print(f"=== {name} ===")
    print(f"rows: {len(df):,}")
    print(f"{ROW} unique: {df[ROW].nunique():,}  range: {df[ROW].min()}..{df[ROW].max()}")
    print(f"{TREE} unique: {df[TREE].nunique():,} range: {df[TREE].min()}..{df[TREE].max()}")
    print()

basic_report(stand, "Stand")
basic_report(xy,    "XY")

# --- counts per row ---
def counts_array(df, row_col=ROW):
    c = df.groupby(row_col).size()
    full = c.reindex(range(c.index.min(), c.index.max() + 1), fill_value=0)
    return full.index.to_numpy(), full.values  # (rows, counts)

rows_stand, cnt_stand = counts_array(stand, ROW)
rows_xy,    cnt_xy    = counts_array(xy,    ROW)

print("Stand row counts (as array):", cnt_stand.tolist())
print("XY row counts    (as array):", cnt_xy.tolist())

# --- quick anomaly flagging on XY counts ---
s = pd.Series(cnt_xy, index=rows_xy)
med = float(s.median())
mad = float((s - med).abs().median()) or 1.0
low_suspicious = s[s < (med - 3*mad)]  
print("\nSuspiciously low XY rows (3*MAD rule):")
print(low_suspicious if not low_suspicious.empty else "None")

# --- concise summary top/bottom counts ---
print("\nXY rows with fewest stems:")
print(s.sort_values().head(8))
print("\nXY rows with most stems:")
print(s.sort_values(ascending=False).head(8))

# --- geometry cross-check---
if 64 in xy[ROW].unique() and 65 in xy[ROW].unique():
    xr64 = xy.loc[xy[ROW]==64, 'X']
    xr65 = xy.loc[xy[ROW]==65, 'X']
    yr   = xy.groupby(ROW)['Y'].median().sort_index()

    x64 = (float(xr64.min()), float(xr64.max()))
    x65 = (float(xr65.min()), float(xr65.max()))
    overlap = max(0.0, min(x64[1], x65[1]) - max(x64[0], x65[0]))
    frac_65_inside_64 = overlap / ( (x65[1]-x65[0]) if (x65[1]>x65[0]) else 1.0 )

    typical_row_step = float(yr.diff().abs().median())
    gap_64_65 = float(yr.loc[65] - yr.loc[64])

    print(f"\nRow 64 X-range: {x64}, Row 65 X-range: {x65}")
    print(f"X-overlap fraction of row 65 inside row 64: {frac_65_inside_64:.2f}")
    print(f"Median Y gap 64→65: {gap_64_65:.3f} (typical step ≈ {typical_row_step:.3f})")

=== Stand ===
rows: 3,253
Row unique: 64  range: 1..64
Tree unique: 60 range: 1..60

=== XY ===
rows: 3,068
Row unique: 65  range: 1..65
Tree unique: 60 range: 1..60

Stand row counts (as array): [54, 54, 55, 57, 54, 51, 53, 55, 54, 52, 55, 55, 55, 44, 46, 45, 43, 42, 46, 46, 45, 45, 45, 42, 45, 53, 55, 56, 58, 55, 51, 58, 57, 60, 56, 48, 55, 53, 54, 42, 46, 43, 43, 45, 41, 45, 44, 41, 43, 46, 45, 56, 56, 57, 57, 55, 55, 56, 58, 55, 53, 53, 56, 55]
XY row counts    (as array): [37, 45, 47, 46, 51, 52, 46, 47, 49, 49, 48, 49, 47, 53, 45, 51, 43, 48, 48, 47, 49, 44, 45, 41, 41, 44, 46, 46, 46, 43, 49, 45, 53, 49, 52, 54, 46, 47, 51, 44, 44, 42, 43, 45, 40, 43, 43, 42, 41, 40, 47, 42, 52, 49, 50, 60, 46, 49, 49, 55, 56, 53, 52, 49, 53]

Suspiciously low XY rows (3*MAD rule):
1    37
dtype: int64

XY rows with fewest stems:
1     37
50    40
45    40
24    41
49    41
25    41
52    42
48    42
dtype: int64

XY rows with most stems:
56    60
61    56
60    55
36    54
33    53
14    53
62 

In [9]:
extra_in_xy   = sorted(set(xy['Row'])   - set(stand['Row']))
missing_in_xy = sorted(set(stand['Row'])- set(xy['Row']))
print("Rows only in XY   :", extra_in_xy)
print("Rows only in Stand:", missing_in_xy)

for r in extra_in_xy:
    print(f"XY count in Row {r}: { (xy['Row']==r).sum() }")


Rows only in XY   : [65]
Rows only in Stand: []
XY count in Row 65: 53
